# Lab 00: Set Up the Lab Environment

**Book chapter:** Appendix C: Lab Setup  
**Estimated time:** 10 minutes

## Overview

Every lab in this book runs in Python against your own Google Cloud project.
This notebook prepares that project once: it installs the client libraries,
signs you in, enables every Google Cloud API that the labs use, and confirms
that Gemini responds. Run it before your first lab, and again whenever you
switch to a new project.

## Learning Objectives

By the end of this lab, you will be able to:

- Install the Google Cloud client libraries used by the labs
- Authenticate a notebook to Google Cloud
- Enable Google Cloud APIs from Python with the Service Usage API
- Verify that a Gemini model responds in your project

## Before You Begin

- A Google Cloud project with billing enabled and the Owner role. Creating the project and linking billing are described in Appendix C.
- A notebook environment: Google Colab (recommended), Colab Enterprise, Vertex AI Workbench, or Jupyter on your own computer with the Google Cloud CLI installed

**Cost:** Enabling APIs is free. The single Gemini request in this notebook costs a fraction of a cent.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Step 1: Install the Client Libraries

The labs use Google's official Python client libraries. This cell installs the
libraries that this notebook needs. Each lab notebook installs its own libraries,
so you can also run any lab on its own.

In [ ]:
%pip install --quiet --upgrade google-genai google-cloud-service-usage google-auth

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

## Step 2: Configure Your Project

Enter your project ID in `PROJECT_ID`. If you use Colab, you can type it in the form
field that appears next to the cell. `MODEL` holds the Gemini model that the labs use;
model IDs change over time, so replace it if Google retires this version.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

## Step 3: Authenticate

Google Cloud client libraries find credentials through **Application Default
Credentials (ADC)**. In Colab, the next cell opens a sign-in window. In other
environments, ADC comes from the `gcloud auth application-default login` command.

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 4: Enable the APIs

Each Google Cloud service must be enabled in a project before it can be used. The
following cell enables all the APIs that the labs in this book use in a single
request through the **Service Usage API**, so that no lab fails because a service
is turned off. Enabling an API is free; you pay only for resources that you create.

| API | Used for |
|---|---|
| aiplatform.googleapis.com | Gemini models, Agent Runtime, and evaluation |
| discoveryengine.googleapis.com | Agent Search data stores and search |
| dialogflow.googleapis.com | State-based conversation flows |
| storage.googleapis.com | Cloud Storage buckets |
| run, cloudbuild, artifactregistry | Deploying agents and MCP servers to Cloud Run |
| sqladmin.googleapis.com | Cloud SQL databases for MCP Toolbox |
| logging, monitoring, cloudtrace, telemetry | Logs, metrics, alerts, and traces |
| modelarmor.googleapis.com | Screening prompts and responses |
| iam, cloudresourcemanager | Service accounts and permissions |

This step can take a minute or two.

In [ ]:
from google.cloud import service_usage_v1

APIS = [
    "aiplatform.googleapis.com", "discoveryengine.googleapis.com",
    "dialogflow.googleapis.com", "storage.googleapis.com",
    "run.googleapis.com", "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com", "sqladmin.googleapis.com",
    "logging.googleapis.com", "monitoring.googleapis.com",
    "cloudtrace.googleapis.com", "telemetry.googleapis.com",
    "modelarmor.googleapis.com", "iam.googleapis.com",
    "cloudresourcemanager.googleapis.com",
]

usage = service_usage_v1.ServiceUsageClient()
operation = usage.batch_enable_services(
    request=service_usage_v1.BatchEnableServicesRequest(
        parent=f"projects/{PROJECT_ID}", service_ids=APIS))
print("Enabling APIs...")
operation.result(timeout=900)
print("Done.")

Now confirm that every API is enabled. The cell lists the services that are enabled
in your project and compares them with the list above.

In [ ]:
enabled = {
    service.config.name
    for service in usage.list_services(
        request=service_usage_v1.ListServicesRequest(
            parent=f"projects/{PROJECT_ID}", filter="state:ENABLED"))
}
missing = [api for api in APIS if api not in enabled]
print(f"{len(APIS) - len(missing)} of {len(APIS)} lab APIs are enabled.")
if missing:
    print("Not yet enabled:", ", ".join(missing))
    print("Wait a minute and run this cell again.")

**Expected result:** `14 of 14 lab APIs are enabled.`

## Step 5: Verify That Gemini Responds

The final check sends one short request to Gemini through the **Google Gen AI SDK**
(`google-genai`). Setting `vertexai=True` sends the request to Google Cloud and bills
it to your project, rather than to a Gemini API key.

In [ ]:
from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
reply = client.models.generate_content(
    model=MODEL, contents="Reply with the single word READY.")
print(f"{MODEL} replied: {(reply.text or '').strip()}")

**Expected result:** the model replies `READY`. If you see an error, use the table
below.

| Error contains | Likely cause | Fix |
|---|---|---|
| `SERVICE_DISABLED` | An API is not enabled yet | Wait a minute and rerun Step 4 |
| `Could not automatically determine credentials` | ADC is not configured | Rerun Step 3, or run `gcloud auth application-default login` |
| `PERMISSION_DENIED` | Your account lacks a role on the project | Grant yourself the Owner role |
| `404` or `not found` for the model | The model ID is retired or misspelled | Set `MODEL` to a current Flash model ID |

## Check Your Understanding

**1. Why does each lab notebook install its own libraries even though this notebook installs some of them?**

<details><summary>Show answer</summary>

So that every lab can run on its own, in a fresh notebook session, without depending on the state of another notebook.

</details>

**2. What does setting `vertexai=True` in `genai.Client` change?**

<details><summary>Show answer</summary>

Requests go to Gemini on Google Cloud, use your project's credentials and quotas, and are billed to your project instead of to a Gemini API key.

</details>

## Summary

Your project is ready for the labs. You installed the client libraries, authenticated
the notebook, enabled the APIs that every lab uses, and confirmed that Gemini responds.
Continue with the Chapter 03 labs in the `Chapter-03` folder.